In [ ]:
%run ../shared_notebook_setup.py

# OpenAI API Roles: Brief Demonstrations

This notebook demonstrates how `system`, `user`, and `assistant` roles influence model output.

Run each example cell and compare outputs.

## Example 1: `user` Only (Baseline)

No `system` message. The model follows only the user request.

In [2]:
prompt = "Explain recursion in 2 sentences."

baseline = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
)

print("BASELINE (user only):")
print(baseline.choices[0].message.content)

BASELINE (user only):
Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursion, allowing the function to solve a problem by breaking it down into smaller instances of the same problem. The recursive function solves the problem by combining the solutions to these smaller instances, ultimately returning the final result.


## Example 2: Add a `system` Role (Tone Control)

The same user prompt, but now a `system` message enforces concise bullet points.

In [3]:
with_system = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {"role": "system", "content": "Be concise. Respond with exactly 2 bullet points."},
        {"role": "user", "content": prompt},
    ],
    temperature=0,
)

print("WITH SYSTEM ROLE:")
print(with_system.choices[0].message.content)

WITH SYSTEM ROLE:
* Recursion is a programming technique where a function calls itself repeatedly until it reaches a base case that stops the recursion.
* The recursive function breaks down a complex problem into smaller sub-problems of the same type, solving them until the solution to the original problem is found.


## Example 3: `assistant` Role Without Earlier `user` Messages

This compares two inputs:
1. full context (`system` + earlier `user` + `assistant` + follow-up `user`)
2. trimmed context (`system` + `assistant` + follow-up `user`)

If outputs are the same, it shows the earlier `user` message was not necessary for this specific continuation.

In [8]:
messages_full = [
    {"role": "system", "content": "You are a helpful tutor. Keep answers to one sentence."},
    {"role": "user", "content": "What is overfitting?"},
    {"role": "assistant", "content": "Overfitting is when a model memorizes training data and generalizes poorly."},
    {"role": "user", "content": "How is that different from underfitting?"},
    {"role": "assistant", "content": "Underfitting is when a model is too simple and misses important patterns in the data."},
    {"role": "user", "content": "Can you keep the explanation beginner-friendly?"},
    {"role": "assistant", "content": "Think of it like studying for a test: memorizing every past question is overfitting, while barely studying at all is underfitting."},
    {"role": "user", "content": "Give me one real-world analogy."},
]

full_resp = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=messages_full,
    temperature=0,
)

full_text = full_resp.choices[0].message.content.strip()
print(full_text)

Overfitting is like a student memorizing every past exam question instead of understanding the underlying concepts.


## Example 4: Conflicting Instructions (`system` vs `user`)

When instructions conflict, `system` typically has higher priority than `user`.

This cell asks for a strict one-word answer in `system`, while the `user` asks for a paragraph.

In [ ]:
conflict = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {"role": "system", "content": "Answer with exactly one word."},
        {"role": "user", "content": "Describe the benefits of regular exercise in a detailed paragraph."},
    ],
    temperature=0,
)

print("CONFLICT CASE:")
print(conflict.choices[0].message.content)